# Day 17: Positional Encoding — Telling the Model About Order

**Goal:** Fix the position-blindness of self-attention by adding location info to embeddings.

### The problem

Self-attention is **permutation-invariant**: it sees a set, not a sequence.

```
"cat sat dog"   and   "dog sat cat"   →   identical attention math!
```

That's obviously broken for language. We need to inject position info.

### Plan for today

1. **Prove** the position-blindness with code (shuffle tokens → same outputs)
2. Build **learned position embeddings** (GPT-2 style)
3. Build **sinusoidal position encoding** (original Transformer)
4. Visualize the position encoding matrix
5. Train an attention LM **with** and **without** position encoding, compare
6. See how position encoding "stamps" each token with its address

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(42)

## 1. PROOF: Attention Is Permutation-Invariant

Let's show this concretely. Build a small attention layer, run it on a sequence, then shuffle the sequence and run it again. The outputs will be **the same** — just reordered.

In [ ]:
# Simple non-causal attention (so we don't have asymmetry from the mask)

class PlainAttention(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.W_q = nn.Linear(dim, dim, bias=False)
        self.W_k = nn.Linear(dim, dim, bias=False)
        self.W_v = nn.Linear(dim, dim, bias=False)
        self.dim = dim
    def forward(self, x):
        Q, K, V = self.W_q(x), self.W_k(x), self.W_v(x)
        scores = Q @ K.transpose(-2, -1) / (self.dim ** 0.5)
        weights = F.softmax(scores, dim=-1)
        return weights @ V

torch.manual_seed(0)
attn = PlainAttention(dim=8)

# Original sequence: 4 tokens
x = torch.randn(1, 4, 8)

# Shuffled order: same tokens, different positions
perm = torch.tensor([2, 0, 3, 1])
x_shuffled = x[:, perm, :]

# Run attention on both
with torch.no_grad():
    out_orig = attn(x)
    out_shuf = attn(x_shuffled)

# If attention is permutation-invariant, out_shuf should equal out_orig[:, perm, :]
print(f"Original out (first row):  {out_orig[0, 0].numpy().round(3)}")
print(f"Shuffled out[1]:           {out_shuf[0, 1].numpy().round(3)}")
print(f"(These SHOULD be the same — token that was at position 0 is now at position 1)")

# Verify they match
expected = out_orig[:, perm, :]
match = torch.allclose(out_shuf, expected, atol=1e-6)
print(f"\nMatch: {match}")
print("\nProof: shuffling tokens just shuffles outputs. Position is INVISIBLE to attention.")

## 2. The Fix: Add Position Info Before Attention

We'll add a position-specific vector to each token embedding, BEFORE feeding into attention.

```
input = token_embedding + position_embedding
```

Same dimension. Just two vectors added together.

In [ ]:
# Method 1: LEARNED position embeddings (GPT-2 style)

class LearnedPositionEmbedding(nn.Module):
    def __init__(self, max_seq_len, embed_dim):
        super().__init__()
        # Just an nn.Embedding indexed by position [0, 1, 2, ...]
        self.pos_emb = nn.Embedding(max_seq_len, embed_dim)
    
    def forward(self, x):
        # x: (batch, seq_len, embed_dim) — token embeddings
        T = x.size(1)
        positions = torch.arange(T, device=x.device)         # [0, 1, ..., T-1]
        pe = self.pos_emb(positions)                          # (T, embed_dim)
        return x + pe                                          # broadcast over batch

# Demo: create token embeddings, then add positions
torch.manual_seed(0)
B, T, D = 1, 5, 8
token_emb = torch.randn(B, T, D)
pos_emb_layer = LearnedPositionEmbedding(max_seq_len=10, embed_dim=D)

x_with_pos = pos_emb_layer(token_emb)
print(f"Original token embeddings shape: {token_emb.shape}")
print(f"After adding position info:      {x_with_pos.shape}")
print(f"(Same shape, but each token now carries position info)")

# Now check: does shuffling change the output?
perm = torch.tensor([2, 0, 3, 1, 4])
token_emb_shuf = token_emb[:, perm, :]

# Apply position encoding to BOTH
out_orig = pos_emb_layer(token_emb)
out_shuf = pos_emb_layer(token_emb_shuf)

# If position matters, these should NOT just be reorderings
expected_if_perm_invariant = out_orig[:, perm, :]
diff = (out_shuf - expected_if_perm_invariant).abs().max().item()
print(f"\nDifference from 'just shuffled': {diff:.4f}")
print(f"(Non-zero! Position encoding made order matter.)")

## 3. Sinusoidal Position Encoding — The Original Trick

The "Attention Is All You Need" paper used a clever fixed encoding instead of learned. For each position `pos` and dim `i`:

```
PE[pos, 2i]   = sin(pos / 10000^(2i/d))
PE[pos, 2i+1] = cos(pos / 10000^(2i/d))
```

Different dimensions oscillate at different frequencies. Like the hands on a clock — fast hand for seconds, slow hand for hours.

In [ ]:
# Build the sinusoidal position encoding

def sinusoidal_pe(max_len, dim):
    """Compute the classic sin/cos position encoding."""
    pe = torch.zeros(max_len, dim)
    pos = torch.arange(0, max_len).unsqueeze(1).float()  # (max_len, 1)
    # div_term: 1, 1/10000^(2/d), 1/10000^(4/d), ...
    div_term = torch.exp(
        torch.arange(0, dim, 2).float() * -(np.log(10000.0) / dim)
    )
    pe[:, 0::2] = torch.sin(pos * div_term)   # even dims: sin
    pe[:, 1::2] = torch.cos(pos * div_term)   # odd dims:  cos
    return pe

# Build a 100-position × 64-dim PE table
PE = sinusoidal_pe(max_len=100, dim=64)
print(f"PE shape: {PE.shape}")
print(f"\nFirst row (position 0):\n{PE[0, :8].numpy().round(3)}")
print(f"\nSecond row (position 1):\n{PE[1, :8].numpy().round(3)}")
print(f"\nNotice: each position has a UNIQUE 'signature' vector.")

In [ ]:
# Visualize: heatmap of the entire encoding + a few dimensions across positions

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Heatmap: positions vs dimensions
im = axes[0].imshow(PE.numpy(), cmap='RdBu_r', aspect='auto', vmin=-1, vmax=1)
plt.colorbar(im, ax=axes[0])
axes[0].set_xlabel('Embedding dimension')
axes[0].set_ylabel('Position in sequence')
axes[0].set_title('Sinusoidal position encoding heatmap\n(rows = positions, cols = dims)')

# Plot a few dimensions across positions to see the frequencies
positions_to_plot = torch.arange(100)
for d, color in zip([0, 2, 8, 32], ['blue', 'green', 'orange', 'red']):
    axes[1].plot(positions_to_plot.numpy(), PE[:, d].numpy(), 
                 label=f'dim {d}', color=color, linewidth=2)

axes[1].set_xlabel('Position')
axes[1].set_ylabel('PE value')
axes[1].set_title('Different dimensions = different frequencies\n(low dims = fast, high dims = slow)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)
axes[1].axhline(y=0, color='black', linewidth=0.5)

plt.tight_layout()
plt.show()

print("Reading these plots:")
print("  LEFT: heatmap. Each row is one position's full encoding vector.")
print("        Vertical bands of color = sinusoidal oscillation pattern.")
print("\n  RIGHT: how a few specific dimensions vary with position.")
print("         Dim 0 oscillates FAST → distinguishes nearby positions.")
print("         Dim 32 oscillates slowly → distinguishes far-apart positions.")
print("\n  Together, this gives each position a unique 'barcode'.")

### Why each position is uniquely identifiable

Let's measure: how SIMILAR are nearby vs far-apart position vectors?

In [ ]:
# Compute pairwise similarity between position encodings

# Normalize each row
PE_normalized = PE / PE.norm(dim=1, keepdim=True)

# Cosine similarity = normalized @ normalized.T
sim = PE_normalized @ PE_normalized.T

plt.figure(figsize=(8, 7))
plt.imshow(sim.numpy(), cmap='RdBu_r', vmin=-1, vmax=1)
plt.colorbar(label='Cosine similarity')
plt.xlabel('Position')
plt.ylabel('Position')
plt.title('Similarity between position vectors\n(diagonal = 1.0; nearby positions more similar)')
plt.tight_layout()
plt.show()

print("The diagonal is 1.0 (each position is identical to itself).")
print("Near-diagonal cells are bright → nearby positions are similar.")
print("Far-from-diagonal cells fade → distant positions are different.")
print("\nThis lets attention learn 'pay attention to nearby tokens' as one pattern.")

## 4. With vs Without Position Encoding — Training Comparison

Train two identical models on Shakespeare. One has position encoding, one doesn't. See how much it matters.

In [ ]:
# Shakespeare corpus (same as Day 14-16)

text = """to be or not to be that is the question
whether tis nobler in the mind to suffer
the slings and arrows of outrageous fortune
or to take arms against a sea of troubles
and by opposing end them to die to sleep
no more and by a sleep to say we end
the heart ache and the thousand natural shocks
that flesh is heir to tis a consummation
devoutly to be wished to die to sleep
to sleep perchance to dream ay there's the rub
for in that sleep of death what dreams may come
when we have shuffled off this mortal coil
must give us pause there's the respect
that makes calamity of so long life"""

chars = sorted(set(text))
vocab_size = len(chars)
char_to_idx = {c: i for i, c in enumerate(chars)}
idx_to_char = {i: c for i, c in enumerate(chars)}
data = torch.tensor([char_to_idx[c] for c in text], dtype=torch.long)

BLOCK_SIZE = 16
BATCH_SIZE = 32
EMBED_DIM = 32
NUM_HEADS = 4

def get_batch():
    starts = torch.randint(0, len(data) - BLOCK_SIZE, (BATCH_SIZE,))
    x = torch.stack([data[s : s + BLOCK_SIZE] for s in starts])
    y = torch.stack([data[s+1 : s+BLOCK_SIZE+1] for s in starts])
    return x, y


# Multi-head attention (from Day 16, abbreviated)
class MultiHeadAttention(nn.Module):
    def __init__(self, embed_dim, num_heads, max_seq_len):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        self.W_q = nn.Linear(embed_dim, embed_dim, bias=False)
        self.W_k = nn.Linear(embed_dim, embed_dim, bias=False)
        self.W_v = nn.Linear(embed_dim, embed_dim, bias=False)
        self.W_o = nn.Linear(embed_dim, embed_dim, bias=False)
        self.register_buffer('mask', torch.tril(torch.ones(max_seq_len, max_seq_len)))
    
    def forward(self, x):
        B, T, C = x.shape
        H, D = self.num_heads, self.head_dim
        Q = self.W_q(x).view(B, T, H, D).transpose(1, 2)
        K = self.W_k(x).view(B, T, H, D).transpose(1, 2)
        V = self.W_v(x).view(B, T, H, D).transpose(1, 2)
        scores = Q @ K.transpose(-2, -1) / (D ** 0.5)
        scores = scores.masked_fill(self.mask[:T, :T] == 0, float('-inf'))
        weights = F.softmax(scores, dim=-1)
        out = (weights @ V).transpose(1, 2).contiguous().view(B, T, C)
        return self.W_o(out)


# Two language models — one WITH pos encoding, one WITHOUT

class LM_NoPosition(nn.Module):
    def __init__(self, vocab_size, embed_dim, num_heads, block_size):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, embed_dim)
        self.attn = MultiHeadAttention(embed_dim, num_heads, block_size)
        self.head = nn.Linear(embed_dim, vocab_size)
    
    def forward(self, idx, targets=None):
        x = self.tok_emb(idx)         # NO position info!
        x = self.attn(x)
        logits = self.head(x)
        loss = None
        if targets is not None:
            B, T, V = logits.shape
            loss = F.cross_entropy(logits.view(B*T, V), targets.view(B*T))
        return logits, loss


class LM_WithPosition(nn.Module):
    def __init__(self, vocab_size, embed_dim, num_heads, block_size):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, embed_dim)
        self.pos_emb = nn.Embedding(block_size, embed_dim)
        self.attn = MultiHeadAttention(embed_dim, num_heads, block_size)
        self.head = nn.Linear(embed_dim, vocab_size)
    
    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok = self.tok_emb(idx)                                  # (B, T, embed)
        pos = self.pos_emb(torch.arange(T, device=idx.device))   # (T, embed)
        x = tok + pos                                             # add position
        x = self.attn(x)
        logits = self.head(x)
        loss = None
        if targets is not None:
            B, T, V = logits.shape
            loss = F.cross_entropy(logits.view(B*T, V), targets.view(B*T))
        return logits, loss

# Sanity-check sizes
torch.manual_seed(0)
m_nopos = LM_NoPosition(vocab_size, EMBED_DIM, NUM_HEADS, BLOCK_SIZE)
m_pos   = LM_WithPosition(vocab_size, EMBED_DIM, NUM_HEADS, BLOCK_SIZE)

print(f"No-position model:   {sum(p.numel() for p in m_nopos.parameters()):,} params")
print(f"With-position model: {sum(p.numel() for p in m_pos.parameters()):,} params")
print(f"Position encoding adds {sum(p.numel() for p in m_pos.parameters()) - sum(p.numel() for p in m_nopos.parameters())} params  (block_size × embed_dim)")

In [ ]:
# Train both models with the same hyperparameters

def train_model(model, steps=2000, lr=0.01):
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    losses = []
    for _ in range(steps):
        xb, yb = get_batch()
        _, loss = model(xb, yb)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
    return losses

print("Training WITHOUT position encoding...")
torch.manual_seed(42)
m_nopos = LM_NoPosition(vocab_size, EMBED_DIM, NUM_HEADS, BLOCK_SIZE)
losses_nopos = train_model(m_nopos)

print("Training WITH position encoding...")
torch.manual_seed(42)
m_pos = LM_WithPosition(vocab_size, EMBED_DIM, NUM_HEADS, BLOCK_SIZE)
losses_pos = train_model(m_pos)

# Smooth and plot
def smooth(arr, w=50):
    return np.convolve(arr, np.ones(w)/w, mode='valid')

plt.figure(figsize=(10, 5))
plt.plot(smooth(losses_nopos), 'r-', linewidth=2, label=f'No position (final {losses_nopos[-1]:.3f})')
plt.plot(smooth(losses_pos), 'b-', linewidth=2, label=f'With position (final {losses_pos[-1]:.3f})')
plt.xlabel('Step (smoothed)')
plt.ylabel('Loss')
plt.title('Effect of position encoding on training')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print(f"\nWithout position: final loss = {losses_nopos[-1]:.3f}")
print(f"With position:    final loss = {losses_pos[-1]:.3f}")
print(f"\nThe model with position encoding does better because it can use ORDER.")
print(f"Without position, attention treats the input as a SET — much harder task.")

## 5. Inspect the LEARNED Position Embeddings

The position embeddings in `LM_WithPosition` were learned by gradient descent. Let's see what patterns emerged.

In [ ]:
# Visualize the learned position embeddings

learned_pos = m_pos.pos_emb.weight.data    # (block_size, embed_dim)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Heatmap
im = axes[0].imshow(learned_pos.numpy(), cmap='RdBu_r', aspect='auto')
plt.colorbar(im, ax=axes[0])
axes[0].set_xlabel('Embedding dimension')
axes[0].set_ylabel('Position')
axes[0].set_title('LEARNED position embeddings\n(after training)')

# Similarity matrix between learned positions
norm = learned_pos / learned_pos.norm(dim=1, keepdim=True)
sim = norm @ norm.T

im2 = axes[1].imshow(sim.numpy(), cmap='RdBu_r', vmin=-1, vmax=1)
plt.colorbar(im2, ax=axes[1])
axes[1].set_xlabel('Position')
axes[1].set_ylabel('Position')
axes[1].set_title('Similarity between learned position vectors')

plt.tight_layout()
plt.show()

print("The model LEARNED these patterns automatically!")
print("Without us telling it, gradient descent figured out useful position representations.")
print("Sinusoidal encoding would give similar patterns — just without learning.")

## 6. Build a Sinusoidal Version Too

Easy to swap in. Same model, but use fixed sin/cos instead of learned.

In [ ]:
class LM_Sinusoidal(nn.Module):
    def __init__(self, vocab_size, embed_dim, num_heads, block_size):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, embed_dim)
        # Precompute sinusoidal PE — non-trainable
        self.register_buffer('pe', sinusoidal_pe(block_size, embed_dim))
        self.attn = MultiHeadAttention(embed_dim, num_heads, block_size)
        self.head = nn.Linear(embed_dim, vocab_size)
    
    def forward(self, idx, targets=None):
        B, T = idx.shape
        x = self.tok_emb(idx) + self.pe[:T]   # broadcasts over batch
        x = self.attn(x)
        logits = self.head(x)
        loss = None
        if targets is not None:
            B, T, V = logits.shape
            loss = F.cross_entropy(logits.view(B*T, V), targets.view(B*T))
        return logits, loss

# Train it
print("Training with SINUSOIDAL position encoding...")
torch.manual_seed(42)
m_sin = LM_Sinusoidal(vocab_size, EMBED_DIM, NUM_HEADS, BLOCK_SIZE)
losses_sin = train_model(m_sin)

# Compare all three
plt.figure(figsize=(11, 5))
plt.plot(smooth(losses_nopos), 'r-', linewidth=2, label=f'No position ({losses_nopos[-1]:.2f})')
plt.plot(smooth(losses_pos), 'b-', linewidth=2, label=f'Learned position ({losses_pos[-1]:.2f})')
plt.plot(smooth(losses_sin), 'g-', linewidth=2, label=f'Sinusoidal ({losses_sin[-1]:.2f})')
plt.xlabel('Step (smoothed)')
plt.ylabel('Loss')
plt.title('Three approaches to position information')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

n_sin = sum(p.numel() for p in m_sin.parameters())
n_learned = sum(p.numel() for p in m_pos.parameters())
print(f"\nLearned PE params: {n_learned}  (PE is trainable)")
print(f"Sinusoidal params: {n_sin}  ({n_learned - n_sin} fewer — PE is FIXED)")
print(f"\nBoth approaches give similar results. Real LLMs vary on this choice.")

---

## Exercises

1. **Stronger shuffling proof:** Take a trained `LM_NoPosition` model. Feed it `"to be"` then `"be to"` (same chars, different order). Predictions should be the same! Verify this.

2. **Bigger model, bigger gap:** Try `EMBED_DIM=64, NUM_HEADS=8`. Is the gap between "with" and "without" position encoding bigger or smaller?

3. **Visualize sinusoidal vs learned (after training):** Plot both PE matrices side by side. Do they look similar?

4. **Initialize learned PE with sinusoidal:** Copy the sinusoidal values into the learned `nn.Embedding`. Does training start with less error?

5. **Only-position model:** Build a model that uses ONLY position embeddings, not token embeddings. What does the loss curve look like? (It should be very high — there's nothing to predict from position alone.)

---

## Key Takeaways

### Why position encoding is mandatory

- Self-attention is **permutation-invariant** — it sees a set, not a sequence
- For language, ORDER matters (sentence meaning depends on it)
- Position encoding "stamps" each token with its address

### Two main approaches

| Method | How | When to use |
|--------|-----|-------------|
| **Learned** (`nn.Embedding`) | Trainable per-position vectors | GPT-2, simple to implement |
| **Sinusoidal** | Fixed sin/cos at varying frequencies | Original Transformer, no extra params, extrapolates |

### The recipe

```python
x = token_embedding(idx) + position_encoding(positions)
out = attention_stack(x)
```

That's it. Add position info **once**, right after token embedding. Every downstream attention layer benefits.

### Where we are

```
Day 13-14: RNN, Bigram LM                ✓
Day 15-16: Self & multi-head attention   ✓
Day 17:    POSITIONAL ENCODING            ✓ ← YOU ARE HERE
Day 18:    Project — full attention generator
Day 19:    TRANSFORMER BLOCK              (layer norm, residual, MLP — the rest)
Day 20+:   Mini GPT — stack of blocks
Day 21-25: Real training, sampling, evaluation
```

### What we have now

You have ALL the core pieces of a transformer:
- Token embedding (Day 11)
- Position encoding (Day 17 ← today)
- Multi-head self-attention (Day 16)
- Causal mask for autoregressive generation (Day 15)
- Cross-entropy loss + sampling (Day 14)

The Day 19+ "transformer block" is mostly **plumbing** around what you already have — layer norm, residual connections, an MLP layer. The HARD ideas are done.

**Tomorrow:** Day 18 — combine all the pieces into a single attention-based text generator and train it end-to-end on a longer corpus. Final boss before transformers.